In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from app.data_providers import provide_dataframe, DataFrameRequest
from processing.fixes import fix_score_column, fix_score_columns_for_all_games
import pandas as pd
import sys

from sklearn.model_selection import GroupShuffleSplit

In [2]:
request = DataFrameRequest(
    apply_preprocessing = True,
    add_computed = True,
    filter_pre_encoding_columns = False,
    encode_for_model = False,
    filter_top_players=True,
    filter_clean= True
)

# pipeline execution
df = provide_dataframe(request)


Repo card metadata block was not found. Setting CardData to empty.


# Train test split

In [3]:
df = df.reset_index(drop=True)

In [4]:
df = df.sort_values("GAME_DATE")

In [5]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, test_idx = next(gss.split(df, groups=df["GAME_ID_x"]))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

In [6]:
# Separate our DataSet into a train and a test DataSets. The test DataSet represent 20% of the database.
#test_df = df.sample(frac=0.2, random_state=42)
#train_df = df.drop(test_df.index)

#print(f"Train size: {len(train_df)}")
#print(f"Test size: {len(test_df)}")

Train size: 430327
Test size: 107582


In [9]:
#train_counts = train_df["PLAYER_ID"].value_counts(normalize=True)
#test_counts = test_df["PLAYER_ID"].value_counts(normalize=True)
train_counts = train_df["PLAYER_ID"].value_counts(normalize=False)
test_counts = test_df["PLAYER_ID"].value_counts(normalize=False)

comparison = pd.concat([train_counts, test_counts], axis=1)
comparison.columns = ["train_share", "test_share"]
comparison = comparison.fillna(0)

print(comparison.sort_values("test_share", ascending=False).head(10))

           train_share  test_share
PLAYER_ID                         
2544.0           40967        9852
977.0            33755        8570
1717.0           28072        7730
1495.0           27641        6848
201142.0         27754        6485
201935.0         25611        6481
201566.0         26781        5954
2548.0           23394        5943
708.0            23593        5664
101108.0         20345        5451


In [10]:
print(comparison['test_share']/(comparison['test_share'] + comparison['train_share']))

PLAYER_ID
2544.0       0.193864
977.0        0.202481
1717.0       0.215910
201142.0     0.189404
1495.0       0.198556
201566.0     0.181885
201935.0     0.201951
708.0        0.193595
2548.0       0.202577
201939.0     0.201082
101108.0     0.211312
2225.0       0.205905
2200.0       0.216177
203507.0     0.192649
203076.0     0.186867
1938.0       0.221773
202695.0     0.196773
203999.0     0.177786
1629029.0    0.198665
203110.0     0.236251
dtype: float64


In [11]:
pd.set_option('display.max_columns', None)

In [12]:
df.head()

,GAME_ID_x,GAME_EVENT_ID,SHOT_TYPE,SHOT_DISTANCE,SHOT_ZONE_RANGE,SHOT_ZONE_BASIC,SHOT_ZONE_AREA,LOC_X,LOC_Y,ACTION_TYPE,HTM,VTM,GAME_DATE,TEAM_ID,PLAYER1_TEAM_ABBREVIATION,PLAYER_ID,scoreHome,scoreAway,shotResult,MINUTES_REMAINING,SECONDS_REMAINING,clock,PLAYER_NAME,PERIOD_x,PLAYER2_ID,PLAYER2_NAME,PLAYER2_TEAM_ABBREVIATION,PLAYER3_ID,PLAYER3_NAME,is_playoffs,PCTIMESTRING,SHOT_MADE_FLAG,IS_HOME,points,pointsHome,pointsAway,scoreHomeBeforeShot,scoreAwayBeforeShot,scoreMargin,scoreMarginBeforeShot,TimeRemainingInPeriod,TotalPlayedTime,TimeRemainingInGame,IsOvertime,OvertimeNumber,IsClutchTime,OPPONENT_INTERFERED,ANGLE,ANGLE_SECTOR,ABS_ANGLE,ANGLE_SIN,ANGLE_COS,MAIN_ACTION_TYPE
448756,29600008.0,281.0,2PT Field Goal,8.0,8-16 ft.,In The Paint (Non-RA),Left Side(L),-42.0,69.0,Jump Shot,MIN,SAS,19961101.0,1.610613e+09,MIN,708.0,50,44,Missed,5.0,25.0,PT05M25.00S,Kevin Garnett,3.0,0.0,NaN,NaN,0.0,NaN,False,5:25,0.0,1,2,0,0,50.0,44.0,6,6.0,325,1835,1045,0,0,0,False,-31.328693,0,31.328693,0.087123,0.996198,Jump
448749,29600008.0,168.0,1PT Free Throw,15.0,8-16 ft.,Mid-Range,Center(C),0.0,15.0,Free Throw,MIN,SAS,19961101.0,1.610613e+09,MIN,708.0,27,31,Missed,6.0,31.0,PT06M31.00S,Kevin Garnett,2.0,0.0,NaN,NaN,0.0,NaN,False,6:31,0.0,1,1,0,0,27.0,31.0,-4,-4.0,391,1049,1831,0,0,0,False,0.000000,0,0.000000,0.000000,1.000000,Other
448762,29600008.0,404.0,2PT Field Goal,12.0,8-16 ft.,Mid-Range,Right Side(R),112.0,58.0,Jump Shot,MIN,SAS,19961101.0,1.610613e+09,MIN,708.0,71,70,Missed,2.0,53.0,PT02M53.00S,Kevin Garnett,4.0,0.0,NaN,NaN,0.0,NaN,False,2:53,0.0,1,2,0,0,71.0,70.0,1,1.0,173,2707,173,0,0,1,False,62.622297,1,62.622297,-0.208025,0.978123,Jump
448751,29600008.0,206.0,2PT Field Goal,14.0,8-16 ft.,Mid-Range,Right Side(R),140.0,-5.0,Jump Shot,MIN,SAS,19961101.0,1.610613e+09,MIN,708.0,35,34,Missed,3.0,13.0,PT03M13.00S,Kevin Garnett,2.0,0.0,NaN,NaN,0.0,NaN,False,3:13,0.0,1,2,0,0,35.0,34.0,1,1.0,193,1247,1633,0,0,0,False,92.045408,2,92.045408,-0.807099,-0.590417,Jump
448754,29600008.0,260.0,2PT Field Goal,16.0,16-24 ft.,Mid-Range,Right Side(R),151.0,67.0,Jump Shot,MIN,SAS,19961101.0,1.610613e+09,MIN,708.0,48,42,Missed,7.0,45.0,PT07M45.00S,Kevin Garnett,3.0,0.0,NaN,NaN,0.0,NaN,False,7:45,0.0,1,2,0,0,48.0,42.0,6,6.0,465,1695,1185,0,0,0,False,66.072727,1,66.072727,-0.099118,-0.995076,Jump


# Save the files

In [13]:
df.to_parquet('../data/processed/processed_20_players.parquet')
train_df.to_parquet('../data/processed/processed_20_players_train.parquet')
test_df.to_parquet('../data/processed/processed_20_players_test.parquet')